# Reproducing Steinberg et al. 2021 (CLMBR), very small scale

Paper: `papers/Steinberg_2021.pdf` — "Language models are an effective representation learning technique for electronic health record data" (CLMBR).

Architecture reproduced here:
1. Group each patient's events into **per-day sets of codes**.
2. **Mean-pool** each day's code embeddings (`EmbeddingBag`).
3. **Concat** day/time features (age, log age, log gap since last day, first-day flag).
4. Feed the day-vector sequence through a single-layer **GRU**.
5. **GELU** + linear head projects each GRU hidden state to next-day code logits.
6. Pretraining objective: predict the *set* of codes on day `i+1` given days `1..i` (multi-label).

Simplifications vs. the paper (kept intentionally small-scale):
- Plain multi-label `BCEWithLogits` loss instead of the paper's hierarchical-sigmoid-over-ontology loss (that needs UMLS/`concept_ancestor` expansion).
- Small vocab (top ~256 codes by frequency), small embedding/hidden sizes, a handful of subjects, a few epochs — enough to see the pretraining loss go down, not to match paper-scale results.

Data source: a small real sample pulled from `ETHOS_PRODUCTION.OMOP_54` in Snowflake.

This notebook is written to be run top-to-bottom later. Nothing has been executed yet.

## Stage 1 — Pull a tiny real OMOP sample from Snowflake

Requires `pip install snowflake-connector-python`. Uses SSO browser auth by default (`authenticator="externalbrowser"`) — swap for your org's usual auth method if different.

Set these env vars before running (or replace inline):
- `SNOWFLAKE_ACCOUNT`
- `SNOWFLAKE_USER`
- `SNOWFLAKE_WAREHOUSE`

First we just list tables under the schema to confirm names before assuming anything.

In [38]:
import os
import snowflake.connector
import csv
import pandas as pd
import random

# Initialize the generator with a fixed integer seed
random.seed(42)

DATABASE = "ETHOS_PRODUCTION"
SCHEMA = "OMOP_54"

conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    authenticator="externalbrowser",
    database=DATABASE,
    schema=SCHEMA,
)

cur = conn.cursor()
cur.execute(f"SHOW TABLES IN SCHEMA {DATABASE}.{SCHEMA}")
existing_tables = {row[1].lower() for row in cur.fetchall()}
print(sorted(existing_tables))

 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://ethoslife.okta.com/app/snowflake/exk1lfx2f6z1dVd7l0h8/sso/saml?SAMLRequest=lZJbc9owEIX%2Fikd9tiUbwkUDZGjIhZa2DJfMtG%2BKvYCCLLlaGQO%2FvjKXTvqQzPTNI5%2Bj72jP9m73uQp2YFEa3SdxxEgAOjWZ1Os%2BWS4ewg4J0AmdCWU09MkBkNwOeihyVfBh6TZ6Br9LQBf4izTy%2BkeflFZzI1Ai1yIH5C7l8%2BG3CU8ixgUiWOdx5GLJUHrWxrmCU1pVVVQ1ImPXNGGMUdalXlVLPpE3iOJjRmGNM6lRV8vev%2BkdRExZs0Z4hSdML8bPUp9H8BHl5SxC%2FrRYTMPpj%2FmCBMPr6%2B6MxjIHOwe7kyksZ5NzAPQJZl%2BHnXa7lUQlhiDQhXGE2lQrJbaQmrwonb828l90BRlVZi39sMajPim2MmuvX47psrN7ZVbmzOYT9%2BX4uNOsqzePR3j6md87dhDGtF%2FvUxI8X6tN6mrHiCWMdV2o80csaYWsG8Y3i7jJkw6Pb6JGo%2FmLBCNfqNTCnZzX1OA2BpVcQWS2TpziiaKgf5NT2G9jtdonq9Yxzp6ztmKbDkU0tG6MnJeGnyLYwX%2BPokff2i8L%2BN13Mh5NjZLpIXgwNhfu%2FcriKD6dyCxcnaQcciHVMMssIPrqlDLVnQXh%2FJ47WwKhgzP1300f%2FAE%3D&RelayState=ver%3A3-hint%3A22325597319909382-ETMsDgAAAaCld%2B1mABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAELdjwsevwBgj6eRxl%2F3RTzgAAACg32xVF8wbZof1yy

 pip install snowflake-connector-python[secure-local-storage]


['care_site', 'concept', 'concept_ancestor', 'concept_class', 'concept_relationship', 'concept_synonym', 'death', 'domain', 'drug_strength', 'dsc_survey', 'ethos_hierarchy_flattened', 'location', 'm_drug_dose_calc', 'master_condition_occurrence', 'master_death', 'master_device_exposure', 'master_drug_exposure', 'master_drug_ingredient_exposure', 'master_measurement', 'master_observation', 'master_observation_enriched', 'master_policy_era', 'master_procedure_occurrence', 'master_visit_occurrence', 'omop_concept', 'omop_concept_cpt4', 'omop_policy_link', 'pack_content', 'person', 'policy_era', 'provider', 'relationship', 'vocabulary']


Pick a small, fixed set of `person_id`s, then dump the core OMOP tables (filtered to those ids, and excluding `ethos_dip` disclosure rows) plus the `concept` vocabulary table (ethos_dip disclosure rows excluded; needed later to map `concept_id` -> standard codes like SNOMED/LOINC/RxNorm) to local CSVs.

If `meds_etl_omop` (Stage 2) complains about a missing table, add it to `CORE_TABLES` and rerun this cell.

In [39]:

N_PATIENTS = 64
OUT_DIR = "omop_extract"
os.makedirs(OUT_DIR, exist_ok=True)

CORE_TABLES = [
    "person", "master_visit_occurrence", "master_condition_occurrence", "master_drug_exposure",
    "master_procedure_occurrence", "master_measurement", "master_observation", "death",
]

# concept_id / source_concept_id columns each master_* table uses. Used to
# exclude ethos_dip (custom disclosure vocab) rows at the source, preventively,
# even though today only master_observation actually references it.
DOMAIN_CONCEPT_COLUMNS = {
    "master_visit_occurrence":      ("visit_concept_id", "visit_source_concept_id"),
    "master_condition_occurrence":  ("condition_concept_id", "condition_source_concept_id"),
    "master_drug_exposure":         ("drug_concept_id", "drug_source_concept_id"),
    "master_procedure_occurrence":  ("procedure_concept_id", "procedure_source_concept_id"),
    "master_measurement":           ("measurement_concept_id", "measurement_source_concept_id"),
    "master_observation":           ("observation_concept_id", "observation_source_concept_id"),
}

def dip_exclusion_clause(concept_col, source_concept_col):
    return (
        f"NOT EXISTS (SELECT 1 FROM {DATABASE}.{SCHEMA}.concept c "
        f"WHERE c.concept_id = {concept_col} AND c.vocabulary_id = 'ethos_dip') "
        f"AND NOT EXISTS (SELECT 1 FROM {DATABASE}.{SCHEMA}.concept c "
        f"WHERE c.concept_id = {source_concept_col} AND c.vocabulary_id = 'ethos_dip')"
    )

cur.execute(f"SELECT person_id FROM {DATABASE}.{SCHEMA}.person LIMIT {N_PATIENTS}")
sample_person_ids = [row[0] for row in cur.fetchall()]
print("sampled person_ids:", sample_person_ids)


def dump_table(table, where_clause=None):
    if table not in existing_tables:
        print(f"skip {table}: not found in schema")
        return
    sql = f"SELECT * FROM {DATABASE}.{SCHEMA}.{table}"
    if where_clause:
        sql += f" WHERE {where_clause}"
    cur.execute(sql)
    cols = [c[0].lower() for c in cur.description]
    rows = cur.fetchall()
    with open(os.path.join(OUT_DIR, f"{table}.csv"), "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(cols)
        writer.writerows(rows)
    print(f"{table}: {len(rows)} rows")


id_list = ",".join(str(pid) for pid in sample_person_ids)
for table in CORE_TABLES:
    where = f"person_id IN ({id_list})"
    if table in DOMAIN_CONCEPT_COLUMNS:
        where += " AND " + dip_exclusion_clause(*DOMAIN_CONCEPT_COLUMNS[table])
    dump_table(table, where_clause=where)

dump_table("concept", where_clause="vocabulary_id IS DISTINCT FROM 'ethos_dip'")  # vocabulary table, excludes ethos_dip disclosure vocab

# meds_etl_omop hardcodes standard OMOP table names (see its `tables` dict in
# meds_etl/omop.py) and has no option to remap them, but it only looks up files
# by name -- it doesn't care what the file is named on disk beyond that lookup.
# Rather than renaming our `master_*` dumps (keeping `master_` as the source of
# truth from Snowflake), add standard-named symlinks alongside them so
# meds_etl_omop can find the data.
MASTER_TO_STANDARD_NAME = {
    "master_visit_occurrence": "visit_occurrence",
    "master_condition_occurrence": "condition_occurrence",
    "master_drug_exposure": "drug_exposure",
    "master_procedure_occurrence": "procedure_occurrence",
    "master_measurement": "measurement",
    "master_observation": "observation",
}
for master_name, standard_name in MASTER_TO_STANDARD_NAME.items():
    src = f"{master_name}.csv"
    dst = os.path.join(OUT_DIR, f"{standard_name}.csv")
    if os.path.exists(os.path.join(OUT_DIR, src)) and not os.path.exists(dst):
        os.symlink(src, dst)
        print(f"linked {dst} -> {src}")

sampled person_ids: [4770393, 4770394, 4770395, 4770396, 4770397, 4770398, 4770399, 4770400, 4770401, 4770402, 4770403, 4770404, 4770405, 4770406, 4770407, 4770408, 4770409, 4770410, 4770411, 4770412, 4770413, 4770414, 4770415, 4770416, 4770417, 4770418, 4770419, 4770420, 4770421, 4770422, 4770423, 4770424, 4770425, 4770426, 4770427, 4770428, 4770429, 4770430, 4770431, 4770432, 4770433, 4770434, 4770435, 4770436, 4770437, 4770438, 4770439, 4770440, 4770441, 4770442, 4770443, 4770444, 4770445, 4770446, 4770447, 4770448, 4770449, 4770450, 4770451, 4770452, 4770453, 4770454, 4770455, 4770456]
person: 64 rows
master_visit_occurrence: 1162 rows
master_condition_occurrence: 714 rows
master_drug_exposure: 689 rows
master_procedure_occurrence: 434 rows
master_measurement: 347 rows
master_observation: 630 rows
death: 0 rows
concept: 7402998 rows


## Stage 2 — Convert OMOP → MEDS

No Stanford `femr_stanford_omop_fixer` step — that's STARR-specific, not needed for a generic OMOP source.

In [40]:
%%bash
# pip install "meds-etl==0.3.11"
rm -rf meds_output/
meds_etl_omop omop_extract/ meds_output/

Generating metadata from OMOP `concept` table


Generating metadata from OMOP `concept` table: 100%|██████████| 1/1 [00:02<00:0it/s]0,  2.29s/it]0,  2.29s/it]
Generating metadata from OMOP `concept_relationship` table: 0it [00:00, ?it/s]ting metadata from OMOP `concept_relationship` table: 0it [00:00, ?it/s]
Extracting dataset metadata: 0it [00:00, ?it/s]


Decompressing OMOP tables, mapping to MEDS Unsorted format, writing to disk...


100%|██████████| 8/8 [00:25<00:00,  3.19s/it]█| 8/8 [00:25<00:00,  3.25s/it]
0it [00:00, ?it/s]


Finished converting dataset to MEDS Unsorted.
Converting from MEDS Unsorted to MEDS...


100%|██████████| 11/11 [00:00<00:00, 45.75it/s]


Collating source table data, shard by shard, to create subject timelines...
(Gathering events into timelines)


100%|██████████| 100/100 [00:00<00:00, 654.10it/s]


...finished converting MEDS Unsorted to MEDS
Deleting temporary directory `meds_output/temp`


verify content generated:

In [41]:
%%bash
tree meds_output

meds_output
├── data
│   ├── data_0.parquet
│   ├── data_10.parquet
│   ├── data_12.parquet
│   ├── data_13.parquet
│   ├── data_14.parquet
│   ├── data_17.parquet
│   ├── data_18.parquet
│   ├── data_19.parquet
│   ├── data_2.parquet
│   ├── data_21.parquet
│   ├── data_22.parquet
│   ├── data_25.parquet
│   ├── data_26.parquet
│   ├── data_29.parquet
│   ├── data_3.parquet
│   ├── data_37.parquet
│   ├── data_38.parquet
│   ├── data_39.parquet
│   ├── data_40.parquet
│   ├── data_43.parquet
│   ├── data_48.parquet
│   ├── data_49.parquet
│   ├── data_5.parquet
│   ├── data_53.parquet
│   ├── data_54.parquet
│   ├── data_57.parquet
│   ├── data_59.parquet
│   ├── data_6.parquet
│   ├── data_62.parquet
│   ├── data_63.parquet
│   ├── data_66.parquet
│   ├── data_67.parquet
│   ├── data_71.parquet
│   ├── data_73.parquet
│   ├── data_74.parquet
│   ├── data_76.parquet
│   ├── data_78.parquet
│   ├── data_79.parquet
│   ├── data_80.parquet
│   ├── data_82.parquet
│   ├── data_83.parquet


In [42]:
from pathlib import Path

data_root = Path(".")

print("Train 0:")
display(pd.read_parquet(data_root / "meds_output/data" / "data_0.parquet").head(6))
print("\nTuning 0:")
display(pd.read_parquet(data_root / "meds_output/data" / "data_10.parquet").head(50))

Train 0:


,subject_id,time,code,numeric_value,end,table,text_value,unit,visit_id
0,4770425,1981-06-16 00:00:00,Gender/M,NaN,NaT,person,NaN,NaN,NaN
1,4770425,1981-06-16 00:00:00,MEDS_BIRTH,NaN,NaT,person,NaN,NaN,NaN
2,4770425,2020-01-29 00:00:00,NDC/31722063231,NaN,2020-02-03 00:00:00,drug_exposure,NaN,NaN,105014922
3,4770425,2020-01-29 23:59:59,NUCC/333600000X,NaN,NaT,visit,NaN,NaN,105014922
4,4770425,2021-01-13 23:59:59,Domain/OMOP generated,NaN,NaT,visit,NaN,NaN,104828259
5,4770425,2021-01-13 23:59:59,ICD10CM/F51.01,NaN,2021-01-13 23:59:59,condition,NaN,NaN,104828259



Tuning 0:


,subject_id,time,code,numeric_value,end,table,text_value,unit,visit_id
0,4770403,1954-05-20,Gender/F,NaN,NaT,person,NaN,NaN,NaN
1,4770403,1954-05-20,MEDS_BIRTH,NaN,NaT,person,NaN,NaN,NaN


## Stage 3 — Convert MEDS → meds_reader extract

`meds_reader.SubjectDatabase` (used below) needs this converted extract, not the raw MEDS parquet directly.

In [43]:
%%bash
#pip install meds_reader #already installed via requirements.txt
rm -rf meds_reader_extract/
meds_reader_convert meds_output/ meds_reader_extract/ --num_threads 4
meds_reader_verify meds_output/ meds_reader_extract/

Test passed!


## Stage 4 — Build a small code vocabulary

Reuse `femr`'s existing stat-gathering functions (`map_statistics` / `agg_statistics` / `convert_statistics_to_msgpack` in `femr/src/femr/models/tokenizer/flat_tokenizer.py`) rather than reinventing vocab building.

**Not** using the `FlatTokenizer` class itself — its `__init__` unconditionally reads `dictionary["is_hierarchical"]`, a key `convert_statistics_to_msgpack` never sets, so instantiating it always raises `KeyError`. Working around it by calling the module-level functions directly.

In [44]:
import sys
sys.path.insert(0, os.path.join(os.getcwd(), "femr", "src"))

import functools
import meds_reader
import femr.models.tokenizer.flat_tokenizer as ft
import femr.pat_utils

db = meds_reader.SubjectDatabase("meds_reader_extract/")

stats = functools.reduce(
    ft.agg_statistics,
    db.map(functools.partial(ft.map_statistics, num_subjects=len(db))),
)

VOCAB_SIZE = 256
vocab_dict = ft.convert_statistics_to_msgpack(stats, vocab_size=VOCAB_SIZE, num_numeric=0)
# Some codes appear twice in vocab_dict["vocab"] (once as a plain code, once
# with a text/numeric value), so a naive `enumerate(vocab_dict["vocab"])` dict
# comprehension collapses those keys but keeps their original (non-contiguous)
# indices -- the resulting dict's max index can then exceed len(code_to_idx),
# which is what EmbeddingBag uses as its table size. Dedup first, then assign
# fresh contiguous indices.
code_to_idx = {
    code: i
    for i, code in enumerate(dict.fromkeys(e["code_string"] for e in vocab_dict["vocab"]))
}

print("vocab size:", len(code_to_idx))

vocab size: 256


## Stage 5 — Group each subject's events into per-day code sets

This is the step the paper describes and neither `femr` nor `hf_ehr` implement — written directly against `meds_reader.Subject.events` (same iteration pattern `flat_tokenizer.map_statistics` already uses).

In [ ]:
import collections
import math


def subject_to_day_sequence(subject, code_to_idx):
    by_day = collections.defaultdict(set)
    for event in subject.events:
        if event.time is not None and event.code in code_to_idx:
            by_day[event.time.date()].add(code_to_idx[event.code])

    days = sorted(by_day)
    birth_date = femr.pat_utils.get_subject_birthdate(subject)

    day_features = []  # [age_days, log1p(age_days), gap_days, log1p(gap_days), is_first_day]
    for i, d in enumerate(days):
        age_days = (d - birth_date.date()).days
        gap_days = (d - days[i - 1]).days if i > 0 else 0
        day_features.append([age_days, math.log1p(max(age_days, 0)),gap_days, math.log1p(max(gap_days, 0)), float(i == 0)])

    code_sets = [by_day[d] for d in days]
    return code_sets, day_features

## Stage 6 — Small CLMBR-replica model + train loop

`EmbeddingBag` (mean-pooled code embedding per day) → concat day features → `GRU` (single layer) → GELU → linear head → per-code sigmoid (`BCEWithLogits`) next-day prediction. `gru_out` at any timestep is the CLMBR patient representation at that point in time.

In [49]:
import torch
from torch import nn


class TinyCLMBR(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_dim=64, num_day_features=4):
        super().__init__()
        self.embed_dim = embed_dim
        self.embed_bag = nn.EmbeddingBag(vocab_size, embed_dim, mode="mean")
        self.gru = nn.GRU(embed_dim + num_day_features, hidden_dim, num_layers=1, batch_first=True)
        self.head = nn.Linear(hidden_dim, vocab_size)

    def forward(self, code_sets, day_features):
        # code_sets: list[LongTensor] (variable-length per day); day_features: FloatTensor [T, num_day_features]
        day_vecs = torch.stack([
            self.embed_bag(cs.unsqueeze(0)).squeeze(0) if len(cs) else torch.zeros(self.embed_dim)
            for cs in code_sets
        ])
        x = torch.cat([day_vecs, day_features], dim=-1).unsqueeze(0)  # [1, T, embed_dim+num_day_features]
        gru_out, _ = self.gru(x)
        return self.head(nn.functional.gelu(gru_out)).squeeze(0)      # [T, vocab_size]

In [50]:
model = TinyCLMBR(vocab_size=len(code_to_idx), num_day_features=5)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

sampled_subjects = [db[pid] for pid in list(db)[:N_PATIENTS]]

N_EPOCHS = 20
for epoch in range(N_EPOCHS):
    total_loss = 0.0
    n_subjects_used = 0
    for subject in sampled_subjects:
        code_sets, day_features = subject_to_day_sequence(subject, code_to_idx)
        if len(code_sets) < 2:
            continue
        n_subjects_used += 1

        logits = model(
            [torch.tensor(sorted(cs), dtype=torch.long) for cs in code_sets[:-1]],
            torch.tensor(day_features[:-1], dtype=torch.float32),
        )

        targets = torch.zeros(len(code_sets) - 1, len(code_to_idx))
        for t, next_day_codes in enumerate(code_sets[1:]):
            targets[t, list(next_day_codes)] = 1.0

        loss = nn.functional.binary_cross_entropy_with_logits(logits, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"epoch {epoch}: loss={total_loss:.4f} (subjects used: {n_subjects_used})")

epoch 0: loss=13.2492 (subjects used: 22)
epoch 1: loss=9.0748 (subjects used: 22)
epoch 2: loss=5.5731 (subjects used: 22)
epoch 3: loss=3.3258 (subjects used: 22)
epoch 4: loss=2.2182 (subjects used: 22)
epoch 5: loss=1.6987 (subjects used: 22)
epoch 6: loss=1.4299 (subjects used: 22)
epoch 7: loss=1.2734 (subjects used: 22)
epoch 8: loss=1.1737 (subjects used: 22)
epoch 9: loss=1.1059 (subjects used: 22)
epoch 10: loss=1.0577 (subjects used: 22)
epoch 11: loss=1.0222 (subjects used: 22)
epoch 12: loss=0.9953 (subjects used: 22)
epoch 13: loss=0.9746 (subjects used: 22)
epoch 14: loss=0.9583 (subjects used: 22)
epoch 15: loss=0.9454 (subjects used: 22)
epoch 16: loss=0.9349 (subjects used: 22)
epoch 17: loss=0.9264 (subjects used: 22)
epoch 18: loss=0.9194 (subjects used: 22)
epoch 19: loss=0.9136 (subjects used: 22)


## Inspect a CLMBR representation

The pretraining loss trending down over the epochs above is the "reproduced the paper, small scale" signal. As a further sanity check, pull representations for 3 patients at their last observed day and compare them — meaningfully different vectors (not near-identical) is evidence the model learned patient-specific signal rather than collapsing to population averages.

In [53]:
N_INSPECT = 10


def get_day_representations(subject):
    """Encoder path (embed_bag -> concat day features -> GRU), same as TinyCLMBR.forward
    minus the final linear head. Returns one representation vector per observed day,
    in chronological order."""
    code_sets, day_features = subject_to_day_sequence(subject, code_to_idx)
    with torch.no_grad():
        day_vecs = torch.stack([
            model.embed_bag(torch.tensor(sorted(cs), dtype=torch.long).unsqueeze(0)).squeeze(0)
            if len(cs) else torch.zeros(model.embed_dim)
            for cs in code_sets
        ])
        x = torch.cat([day_vecs, torch.tensor(day_features, dtype=torch.float32)], dim=-1).unsqueeze(0)
        gru_out, _ = model.gru(x)
    return gru_out[0]  # shape: [n_days_observed, hidden_dim]


last_day_representations = []
for i, subject in enumerate(sampled_subjects[:N_INSPECT]):
    # [n_days, hidden_dim]: one vector per calendar day this patient has an event,
    # hidden_dim=64 dims each (set by TinyCLMBR's hidden_dim=64 default).
    day_reprs = get_day_representations(subject)

    # The representation at their *most recent* observed day -- this is "the" CLMBR
    # representation people usually mean: a fixed-size (64-d) summary of everything
    # the GRU has seen for this patient (diagnoses, meds, labs, demographics, visit
    # timing) up to and including that day. Earlier rows of day_reprs are the same
    # kind of summary but truncated to an earlier point in the patient's timeline --
    # useful if you want "patient state as of day X" rather than "as of today".
    last_repr = day_reprs[-1]
    last_day_representations.append(last_repr)

    print(f"patient {i} (subject_id={subject.subject_id})")
    print(f"  days observed: {day_reprs.shape[0]}")
    print(f"  all-days representation shape: {tuple(day_reprs.shape)}   # [n_days, hidden_dim]")
    print(f"  last-day representation shape: {tuple(last_repr.shape)}   # (hidden_dim,) = one 64-d vector")
    print(f"  {last_repr}")
    print()

# Sanity check: three different patients should get three *different* vectors. If
# cosine similarity is ~1.0 across the board, the model collapsed to predicting
# population-average next-day codes instead of learning patient-specific history --
# the representations wouldn't be useful for anything downstream (e.g. feeding into
# a classifier for a clinical outcome, which is the whole point of CLMBR).
stacked = torch.stack(last_day_representations)  # [3, hidden_dim]
similarity = torch.nn.functional.cosine_similarity(stacked.unsqueeze(1), stacked.unsqueeze(0), dim=-1)
print("pairwise cosine similarity between the 3 patients' last-day representations:")
print(similarity)
# similarity shape: [3, 3]. Diagonal is always 1.0 (a vector compared to itself);
# off-diagonal entries meaningfully below 1.0 indicate the patients are
# distinguishable in representation space, not collapsed to the same point.

patient 0 (subject_id=4770393)
  days observed: 1
  all-days representation shape: (1, 64)   # [n_days, hidden_dim]
  last-day representation shape: (64,)   # (hidden_dim,) = one 64-d vector
  tensor([-0.6674, -0.7800, -0.7557,  0.8720, -0.7953, -0.7258,  0.9077, -0.7765,
         0.8765,  0.8979, -0.6953,  0.9731, -0.7692,  0.9540, -0.7316,  0.9623,
         0.9764,  0.8917, -0.7399,  0.9749,  0.9128, -0.7574, -0.6532,  0.9671,
        -0.7417, -0.7861,  0.8859, -0.6705, -0.7440, -0.6277, -0.7706,  0.8890,
         0.8924, -0.7689,  0.9593, -0.6238,  0.9616,  0.9158,  0.8788,  0.8969,
         0.8848, -0.7246, -0.7749,  0.9769,  0.9354, -0.6518, -0.6853,  0.9659,
         0.9700,  0.9078, -0.6933,  0.9636,  0.9141,  0.9398,  0.9633, -0.6979,
         0.9152,  0.9029, -0.7546,  0.9588,  0.9719, -0.6186, -0.7367,  0.8752])

patient 1 (subject_id=4770394)
  days observed: 1
  all-days representation shape: (1, 64)   # [n_days, hidden_dim]
  last-day representation shape: (64,)   # (hidde